# 🚀 LSTM Product Training - MEDIUM (Colab Pro)

**Configuración:** 120 días → 7 días (más contexto)

**GPU A100:** ~1.5-2h ⚡

---

## 📌 ANTES DE EMPEZAR:
1. Runtime → Change runtime type → GPU → **A100 GPU**
2. Guardar configuración

In [ ]:
# 1. Verificar GPU
import tensorflow as tf
import gc

print("="*80)
print("PRODUCT MEDIUM: 120→7 días")
print("="*80)

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"\n✅ GPU: {gpus[0].name}")
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
else:
    print("\n⚠️ NO GPU - Ve a Runtime → Change runtime type")

print("="*80)

In [ ]:
# 2. Instalar dependencias
!pip install -q openpyxl seaborn
print("✅ Dependencias instaladas")
gc.collect()

In [ ]:
# 3. Subir archivos
from google.colab import files
import os

print("📤 Sube 'online_retail_2.xlsx'")
uploaded1 = files.upload()

print("\n📤 Sube 'train_products_temporal.py'")
uploaded2 = files.upload()

# Crear directorios
!mkdir -p /content/data/processed
!mkdir -p /content/models/temporal/products/medium

# Mover archivos
!mv online_retail_2.xlsx /content/data/processed/
!mv train_products_temporal.py /content/

print("\n✅ Archivos listos")
gc.collect()

In [ ]:
# 4. Importar script
import sys
sys.path.append('/content')

from train_products_temporal import ProductTemporalAnalyzer, ProductTemporalConfig

print("✅ Script importado")
print(f"📊 Config: {ProductTemporalConfig.MEDIUM['window_days']}→{ProductTemporalConfig.MEDIUM['forecast_days']}d")
gc.collect()

In [ ]:
# 5. Preparar datos
import warnings
from datetime import datetime
warnings.filterwarnings('ignore')

start_time = datetime.now()
print(f"⏰ Inicio: {start_time}\n")

analyzer = ProductTemporalAnalyzer(
    data_path='/content/data/processed/online_retail_2.xlsx',
    output_dir='/content/models/temporal/products'
)

analyzer.load_and_preprocess_data()
gc.collect()

analyzer.select_products(min_transactions=20, top_n=50)
gc.collect()

print(f"\n✅ {len(analyzer.products)} productos listos para entrenar")

In [ ]:
# 6. ENTRENAR MEDIUM
import time

print("="*70)
print("ENTRENAMIENTO MEDIUM (120→7 días)")
print("="*70)

t0 = time.time()
model, history, metrics = analyzer.train_horizon_model(ProductTemporalConfig.MEDIUM)
mins = (time.time() - t0) / 60

print(f"\n✅ COMPLETADO en {mins:.1f} min")
print(f"   MAE: {metrics['mae']:.2f}")
print(f"   RMSE: {metrics['rmse']:.2f}")

# Guardar resultados
with open('/content/models/temporal/products/medium/RESULTADOS.txt', 'w') as f:
    f.write(f"MEDIUM - RESULTADOS\n")
    f.write(f"Tiempo: {mins:.1f} min\n")
    f.write(f"MAE: {metrics['mae']:.2f}\n")
    f.write(f"RMSE: {metrics['rmse']:.2f}\n")

total = (datetime.now() - start_time).total_seconds() / 60
print(f"\n⏰ Tiempo total: {total:.1f} min")

In [ ]:
# 7. Comprimir y descargar
!cd /content/models/temporal/products && zip -r medium.zip medium/

print("✅ Modelos comprimidos")
print("\n📥 Descarga el archivo:")
print("   Panel izquierdo → Files → models/temporal/products/medium.zip")
print("   Click derecho → Download")

!ls -lh /content/models/temporal/products/*.zip